In [ ]:

import pandas as pd

# 1. Оптимізуємо типи при читанні
dtypes = {
    'country': 'category',
    'event_name': 'category',
    'platform': 'category',
    'category': 'category',
    'operating_system_version': 'category',
    'app_info_version': 'category'
}
events_file = r'C:\Users\pinko\Documents\dell files\desktop\analytical project\data\events_ios_short_sample_1000'
assets_file = r'C:\Users\pinko\Documents\dell files\desktop\analytical project\data\dim_asset_202606181654.csv'
events = pd.read_csv(events_file, dtype = dtypes)
assets = pd.read_csv(assets_file)
products_file = r'C:\Users\pinko\Documents\dell files\desktop\analytical project\data\dim_product_202606181655.csv'
products = pd.read_csv(products_file)
# 2. Обробка дати та оптимізація числових типів
events['event_datetime'] = pd.to_datetime(events['event_datetime'], errors="coerce")
events["iso_weekday"] = (events['event_datetime'].dt.dayofweek + 1).astype('int8')
events["hour"] = events['event_datetime'].dt.hour.astype('int8')
events.drop(columns=['event_datetime'], inplace=True)

# 3. Мапінг регіонів
regions = {
    "Ukraine": ["Ukraine"],
    "Europe": [
        "Poland", "Germany", "Czechia", "United Kingdom", "Netherlands", "Spain", "Italy", "France",
        "Slovakia", "Belgium", "Denmark", "Sweden", "Austria", "Hungary", "Ireland", "Romania",
        "Lithuania", "Switzerland", "Bulgaria", "Norway", "Finland", "Estonia", "Portugal", "Moldova",
        "Greece", "Croatia", "Latvia", "Albania", "Luxembourg", "Slovenia", "Montenegro", "Russia",
        "Iceland", "Malta", "Serbia", "Bosnia & Herzegovina", "Monaco", "Isle of Man", "North Macedonia",
        "Belarus", "Svalbard & Jan Mayen"
    ],
    "Asia": [
        "Indonesia", "Türkiye", "Cyprus", "Israel", "United Arab Emirates", "Georgia", "Iran", "Japan",
        "Kazakhstan", "Saudi Arabia", "Qatar", "Vietnam", "Uzbekistan", "Iraq", "South Korea",
        "Bangladesh", "Azerbaijan", "Thailand", "Singapore", "China", "Jordan", "Cambodia",
        "Kyrgyzstan", "Armenia", "Malaysia", "Palestine", "Hong Kong", "Turkmenistan", "Maldives",
        "India", "Philippines", "Taiwan", "Syria", "Oman", "Kuwait", "Myanmar (Burma)", "Tajikistan",
        "Nepal", "Lebanon", "Bahrain", "Pakistan", "Laos"
    ],
    "North America": [
        "United States", "Canada", "Mexico", "Guatemala", "Dominican Republic", "Costa Rica",
        "Martinique", "Honduras"
    ],
    "South America": [
        "Brazil", "Colombia", "Ecuador", "Peru", "Chile", "French Guiana", "Argentina"
    ],
    "Africa": [
        "Egypt", "Libya", "Ghana", "South Africa", "Tunisia", "Nigeria", "Côte d’Ivoire",
        "South Sudan", "Zimbabwe", "Morocco", "Cape Verde", "Senegal", "Algeria", "Zambia",
        "Uganda", "Mauritania", "Guinea"
    ],
    "Australia": ["Australia", "New Zealand"]
}

region_map = {country: region for region, countries in regions.items() for country in countries}
events["region"] = events["country"].astype(str).map(region_map).fillna("Other").astype('category')
events.drop(columns=['country'], inplace=True)

# 4. One-Hot Encoding
events_encoded = pd.get_dummies(
    events,
    columns=['event_name', 'platform', 'category'],
    prefix='',
    prefix_sep='',
    dtype='int8',
    sparse=True
)

events_encoded = events_encoded.merge(assets, on='asset_id', how='left')
events_encoded.drop(columns=['asset_id'], inplace=True)
events_encoded = events_encoded.merge(products, left_on='package_id', right_on='product_id', how='left')
events_encoded.drop(columns=['product_id', 'product_name'], inplace=True)

events_encoded = pd.get_dummies(
    events_encoded,
    columns=['payment_type', 'asset_type'],
    prefix='',
    prefix_sep='',
    dtype='int8',
    sparse=True
)

cols_to_drop = ['event_date', 'package_id']
events_encoded.drop(columns=[c for c in cols_to_drop if c in events_encoded.columns], inplace=True)

# 5. Розділяємо колонки на числові (0/1) та текстові
candidate_cols = [
    'age_verification_click', 'catfish_click', 'catfish_display', 'first_open',
    'login_success', 'mainpage_banner_click', 'mainpage_banner_show', 'play_click',
    'pop_up_click', 'pop_up_display', 'screen_view_custom', 'session_start',
    'thank_you_page_open', 'vod_watchlist_add', 'IOS', 'mobile', 'tablet',
    'major_company', 'est', 'svod', 'tvod', 'channel', 'content_area',
    'content_group', 'crew', 'episode', 'movie', 'series', 'sport_event'
]

numeric_binary = []
text_cols = []

for col in candidate_cols:
    if col in events_encoded.columns:
        if pd.api.types.is_numeric_dtype(events_encoded[col]):
            numeric_binary.append(col)
        else:
            text_cols.append(col)

events_encoded[numeric_binary] = events_encoded[numeric_binary].fillna(0).astype('int32')
for col in candidate_cols:
    if col not in events_encoded.columns:
        events_encoded[col] = 0
# 6. УМОВА ПОКУПКИ
events_encoded['product_price'] = pd.to_numeric(events_encoded['product_price'], errors='coerce').fillna(0)
events_encoded['purchase_revenue'] = events_encoded['product_price'] * events_encoded['thank_you_page_open']

# 7. словник агрегації
agg_dict = {col: 'sum' for col in numeric_binary}
for col in text_cols:
    agg_dict[col] = 'count'

agg_dict['purchase_revenue'] = 'sum'

for col in ['iso_weekday', 'hour']:
    if col in events_encoded.columns:
        agg_dict[col] = 'nunique'

# 8. Універсальна функція для підрахунку основного значення та кількості унікальних
def get_category_stats(df, col_name, main_out_name, unique_out_name):
    counts = (
        df.groupby(['user_pseudo_id', col_name], observed=True, sort=False)
        .size()
        .reset_index(name='cnt')
    )
    counts.sort_values(['user_pseudo_id', 'cnt'], ascending=[True, False], inplace=True)
    return counts.groupby('user_pseudo_id', observed=True, sort=False).agg(
        **{
            main_out_name: (col_name, 'first'),
            unique_out_name: (col_name, 'count')
        }
    )

region_summary = get_category_stats(events_encoded, 'region', 'main_region', 'unique_regions_count')
os_summary = get_category_stats(events_encoded, 'operating_system_version', 'main_os_version', 'unique_os_versions_count')
app_summary = get_category_stats(events_encoded, 'app_info_version', 'main_app_version', 'unique_app_versions_count')

# 9. Основна агрегація лише по user_pseudo_id
gb = events_encoded.groupby('user_pseudo_id', observed=True, sort=False)
user_df = gb.agg(agg_dict)

user_df['total_events_count'] = gb.size()
user_df = user_df.join([region_summary, os_summary, app_summary])
user_df.to_csv(r'C:\Users\pinko\Documents\dell files\desktop\analytical project\data\user_df_sample.csv')
